# 3주차: 모델은 어떻게 학습할까?

## 오늘의 목표
- 모델의 **예측(prediction)** 이 무엇인지 알아봅니다.
- 예측이 얼마나 틀렸는지 **loss** 로 계산합니다.
- **gradient** 를 보고 값을 어느 방향으로 고칠지 결정합니다.
- **optimizer** 로 값을 조금씩 고치는 과정을 반복합니다.
- 반복할수록 loss가 줄어드는 것을 그래프로 확인합니다.

> 핵심 흐름: 예측 → loss 계산 → gradient 계산 → 값 수정 → 반복(epoch)


## 1. 지난 시간 복습: 데이터 준비

2주차에 만든 코드입니다. 그대로 실행하고 질문에 답해 보세요.

1. `images_batch.shape`은 무엇이 나올까요?
2. `batch_size=16`은 무슨 뜻이었나요?


In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32) / 16.0
y = torch.tensor(digits.target, dtype=torch.long)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

images_batch, labels_batch = next(iter(loader))
print('images batch shape:', images_batch.shape)
print('labels batch shape:', labels_batch.shape)

## 2. 아주 쉬운 문제로 시작하기: 예측

공부 시간 `x`와 시험 점수 `y`가 있습니다.

| 공부 시간 x | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| 점수 y | 2 | 4 | 6 | 8 | 10 |

모델은 `예측 = w × x` 라는 아주 단순한 모양입니다.
`w`는 모델이 학습해야 하는 숫자이고, 이것을 **가중치(weight)** 또는 **파라미터**라고 부릅니다.

처음에는 `w = 0.5`로 아무렇게나 시작합니다. 예측이 어떻게 나올지 먼저 손으로 계산해 보세요.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
y_true = torch.tensor([2.0, 4.0, 6.0, 8.0, 10.0])

w = 0.5

# 모델의 예측을 계산해 보세요.
y_pred = ???

print('예측:', y_pred)
print('정답:', y_true)

## 3. 얼마나 틀렸을까? loss 계산하기

**loss**는 예측이 정답에서 얼마나 멀리 떨어져 있는지를 나타내는 숫자입니다. loss가 작을수록 좋은 모델입니다.

오늘은 **평균 제곱 오차(MSE)** 를 사용합니다.
1. 오차 = 예측 − 정답
2. 오차를 제곱합니다. (음수를 없애고, 큰 실수를 더 크게 벌줍니다)
3. 평균을 냅니다.


In [ ]:
error = y_pred - y_true
print('오차:', error)

# 오차를 제곱한 뒤 평균을 내 보세요. (힌트: ** 2, .mean())
loss = ???

print('loss:', loss.item())

## 4. w를 직접 바꿔 보기

`w`를 여러 값으로 바꾸면 loss가 어떻게 변하는지 관찰해 봅시다.
먼저 직접 몇 개 넣어 보고, 아래 그래프에서 전체 모양을 확인하세요.


In [ ]:
def compute_loss(w):
    y_pred = w * x
    return ((y_pred - y_true) ** 2).mean().item()

# 여러 w 값을 직접 넣어 보세요.
for w_try in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    print(f'w = {w_try:.1f}  ->  loss = {compute_loss(w_try):.2f}')

In [ ]:
w_values = torch.linspace(-1, 5, 100)
loss_values = [compute_loss(w_v.item()) for w_v in w_values]

plt.figure(figsize=(6, 4))
plt.plot(w_values, loss_values)
plt.scatter([0.5], [compute_loss(0.5)], color='red', zorder=3, label='start w=0.5')
plt.xlabel('w')
plt.ylabel('loss')
plt.title('Loss changes with w')
plt.legend()
plt.grid(True)
plt.show()

## 5. gradient: 어느 방향으로 고쳐야 할까?

PyTorch는 loss를 줄이려면 `w`를 어느 방향으로 얼마나 바꿔야 하는지 **자동으로 계산**해 줍니다.

- `requires_grad=True` : "이 숫자는 학습할 값이니 기울기를 계산해 줘"
- `loss.backward()` : 기울기(gradient)를 계산
- `w.grad` : 계산된 기울기

**규칙**: gradient가 음수(−)면 w를 키우고, 양수(+)면 w를 줄이면 loss가 작아집니다.


In [ ]:
w = torch.tensor(0.5, requires_grad=???)

y_pred = w * x
loss = ((y_pred - y_true) ** 2).mean()

# gradient를 계산하는 함수를 호출하세요.
loss.???()

print('loss:', loss.item())
print('gradient:', w.grad.item())

## 6. w를 한 걸음 고치기

이제 gradient의 **반대 방향**으로 w를 조금 이동시킵니다.

```text
새로운 w = w - learning_rate × gradient
```

`learning_rate`(학습률)는 한 번에 얼마나 크게 걸을지 정하는 **보폭**입니다.


In [ ]:
learning_rate = 0.01

with torch.no_grad():
    # w를 gradient 반대 방향으로 한 걸음 이동시키세요.
    w -= ??? * ???

print('수정된 w:', w.item())

# 수정된 w로 다시 loss를 계산해 봅니다.
new_loss = ((w * x - y_true) ** 2).mean()
print('새 loss:', new_loss.item())

## 7. 반복하기: 이것이 학습입니다

예측 → loss → gradient → 수정 을 여러 번 반복해 봅시다.

주의: PyTorch는 gradient를 **누적**하기 때문에 매번 `w.grad.zero_()`로 0으로 초기화해야 합니다.


In [ ]:
w = torch.tensor(0.5, requires_grad=True)
learning_rate = 0.01
loss_history = []

for step in range(20):
    # 1. 예측
    y_pred = w * x
    # 2. loss 계산
    loss = ((y_pred - y_true) ** 2).mean()
    # 3. gradient 계산
    loss.backward()
    # 4. w 수정
    with torch.no_grad():
        w -= learning_rate * w.grad
    # 5. gradient 초기화
    w.grad.zero_()

    loss_history.append(loss.item())
    print(f'step {step + 1:2d} | w = {w.item():.4f} | loss = {loss.item():.4f}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, 21), loss_history, marker='o')
plt.xlabel('step')
plt.ylabel('loss')
plt.title('Loss during training')
plt.grid(True)
plt.show()

## 8. 실험: learning rate(보폭)를 바꾸면?

아래 `learning_rate` 값을 바꿔 가며 실행해 보세요. 실행 전에 결과를 예상해서 표에 적어 보세요.

| learning rate | 예상 | 실제 결과 |
|---|---|---|
| 0.001 | | |
| 0.01 | | |
| 0.05 | | |
| 0.1 | | |


In [ ]:
def train_w(learning_rate, steps=20):
    w = torch.tensor(0.5, requires_grad=True)
    history = []
    for step in range(steps):
        loss = ((w * x - y_true) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= learning_rate * w.grad
        w.grad.zero_()
        history.append(loss.item())
    return w.item(), history

plt.figure(figsize=(7, 4))
for lr in [0.001, 0.01, 0.05, 0.1]:
    final_w, history = train_w(lr)
    print(f'lr = {lr:<6} -> final w = {final_w:.4f}, final loss = {history[-1]:.4f}')
    plt.plot(history, label=f'lr={lr}')

plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss (log scale)')
plt.title('Effect of learning rate')
plt.legend()
plt.grid(True)
plt.show()

## 9. optimizer: 수정 작업을 PyTorch에게 맡기기

지금까지 `w -= learning_rate * w.grad`와 `w.grad.zero_()`를 직접 썼습니다.
PyTorch에는 이 일을 대신해 주는 **optimizer**가 있습니다.

| 직접 작성 | optimizer 사용 |
|---|---|
| `w.grad.zero_()` | `optimizer.zero_grad()` |
| `w -= lr * w.grad` | `optimizer.step()` |


In [ ]:
w = torch.tensor(0.5, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.01)

for step in range(20):
    optimizer.???()          # 1. gradient 초기화
    y_pred = w * x                # 2. 예측
    loss = ((y_pred - y_true) ** 2).mean()  # 3. loss
    loss.???()                # 4. gradient 계산
    optimizer.???()           # 5. w 수정

print('최종 w:', w.item())
print('최종 loss:', loss.item())

## 10. 진짜 데이터로 학습하기: 손글씨 숫자 분류

이제 같은 원리로 손글씨 숫자 이미지를 분류하는 모델을 학습시켜 봅시다.

- **모델**: `nn.Linear(64, 10)` — 8×8=64개의 픽셀을 받아 0~9 각 숫자의 점수 10개를 내놓습니다. (모델 내부는 4주차에 자세히 배웁니다.)
- **loss**: `nn.CrossEntropyLoss()` — 분류 문제용 오답 점수입니다. 정답 숫자의 점수가 낮으면 loss가 커집니다.
- **epoch**: 전체 학습 데이터를 한 번 다 사용하는 것

먼저 데이터를 학습용과 시험용으로 나눕니다.


In [ ]:
import torch.nn as nn
from torch.utils.data import random_split

torch.manual_seed(0)

train_dataset, test_dataset = random_split(dataset, [1500, 297])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=297)

print('학습 데이터:', len(train_dataset))
print('시험 데이터:', len(test_dataset))
print('1 epoch당 batch 개수:', len(train_loader))

In [ ]:
model = nn.Linear(64, 10)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 학습 전 예측 확인
images_batch, labels_batch = next(iter(test_loader))
with torch.no_grad():
    scores = model(images_batch.reshape(-1, 64))
    predictions = scores.argmax(dim=1)

print('점수 shape:', scores.shape)
print('학습 전 정확도:', (predictions == labels_batch).float().mean().item())
print('학습해야 할 숫자(파라미터) 개수:', sum(p.numel() for p in model.parameters()))

In [ ]:
loss_history = []

for epoch in range(10):
    epoch_loss = 0

    for images_batch, labels_batch in train_loader:
        inputs = images_batch.reshape(-1, 64)

        # 학습 루프 5단계를 완성하세요.
        optimizer.???()
        scores = model(inputs)
        loss = loss_fn(scores, labels_batch)
        loss.???()
        optimizer.???()

        epoch_loss += loss.item()

    average_loss = epoch_loss / len(train_loader)
    loss_history.append(average_loss)
    print(f'epoch {epoch + 1:2d} | loss = {average_loss:.4f}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, 11), loss_history, marker='o')
plt.xlabel('epoch')
plt.ylabel('average loss')
plt.title('Digits classifier training loss')
plt.grid(True)
plt.show()

images_batch, labels_batch = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images_batch.reshape(-1, 64)).argmax(dim=1)

accuracy = (predictions == labels_batch).float().mean().item()
print(f'학습 후 시험 정확도: {accuracy * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))

for i, ax in enumerate(axes.flatten()):
    ax.imshow(images_batch[i], cmap='gray')
    color = 'blue' if predictions[i] == labels_batch[i] else 'red'
    ax.set_title(f'pred {predictions[i].item()} / true {labels_batch[i].item()}', color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 11. 오늘 정리

다음 질문에 자신의 말로 답해 보세요.

1. 모델의 **파라미터(w)** 는 무엇인가요?
2. **loss**는 무엇을 나타내나요? 클수록 좋나요, 작을수록 좋나요?
3. **gradient**는 무엇을 알려주나요?
4. **learning rate**가 너무 크면 어떤 일이 생기나요?
5. 학습 루프의 5단계를 순서대로 말해 보세요.
6. **epoch**와 **batch**의 차이는 무엇인가요?

내 답:
- 1.
- 2.
- 3.
- 4.
- 5.
- 6.


## 12. 도전 과제 (시간이 남으면)

### 도전 1. 두 개의 파라미터 학습하기
정답 규칙이 `y = 2 × x + 1` 일 때, 모델 `예측 = w × x + b` 의 `w`와 `b`를 모두 학습시켜 보세요.


In [ ]:
y_true2 = 2 * x + 1

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.02)

for step in range(???):
    optimizer.zero_grad()
    y_pred = ???
    loss = ((y_pred - y_true2) ** 2).mean()
    loss.backward()
    optimizer.step()

print('w:', w.item(), ' b:', b.item(), ' loss:', loss.item())

### 도전 2. 숫자 분류 모델 설정 바꾸기
10번 셀로 돌아가 다음을 하나씩 바꿔 보고 결과를 표에 기록하세요. (모델을 다시 만드는 셀부터 다시 실행해야 합니다!)

| 바꾼 것 | 값 | 마지막 loss | 시험 정확도 |
|---|---|---|---|
| 기본 | lr=0.1, epoch=10, batch=32 | | |
| learning rate | 0.01 | | |
| learning rate | 1.0 | | |
| epoch | 30 | | |
| batch_size | 4 | | |

### 도전 3. 생각해 보기
- batch_size를 4로 줄이면 1 epoch당 w 수정 횟수는 몇 번이 될까요?
- loss가 계속 줄어들면 시험 정확도도 무조건 계속 올라갈까요?

### 도전 4. 틀린 이미지 찾기
시험 데이터에서 모델이 틀린 이미지만 모아 보세요.


In [ ]:
wrong = torch.where(predictions != labels_batch)[0]
print('틀린 개수:', len(wrong))

# 틀린 이미지들을 최대 6개까지 그려 보세요.
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for index, ax in zip(wrong[:6], axes):
    ax.imshow(images_batch[index], cmap='gray')
    ax.set_title(f'pred {???} / true {???}', fontsize=9)
    ax.axis('off')
plt.show()

## 다음 시간 예고

오늘은 `nn.Linear`를 '이미 만들어진 상자'로 사용했습니다.
4주차에는 이 상자 안을 열어서 **신경망(nn.Module)** 을 직접 설계해 봅니다.
